In [59]:
import pandas as pd
import os

raw_dir = r"C:\Users\mingar\Downloads\Finance\homework\ex03\data_raw"

files = [
    "balance_sheet.csv",
    "income_stmt.csv",
    "cashflow.csv",
    "ownership.csv",
    "industry.csv",
    "m2.csv",
    "st_flag_2010-2014.csv",
    "st_flag_2015-2019.csv",
    "st_flag_2020-2024.csv",
    "st_flag_2025.csv",
]

for f in files:
    path = os.path.join(raw_dir, f)
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"文件: {f}")
        df = pd.read_csv(path, nrows=3)
        print("形状:", df.shape)
        print("列名:", list(df.columns))
        print(df.head(3))
    else:
        print(f"❌ 文件不存在: {f}")


文件: balance_sheet.csv
形状: (3, 8)
列名: ['Stkcd', 'ShortName', 'Accper', 'Typrep', 'A001212000', 'A001000000', 'A002000000', 'A003000000']
   Stkcd ShortName      Accper Typrep    A001212000    A001000000  \
0      1      深发展A  2010-01-01      A  1.714461e+09  5.878110e+11   
1      1      深发展A  2010-03-31      A  1.918952e+09  6.199276e+11   
2      1      深发展A  2010-06-30      A  2.133862e+09  6.243982e+11   

     A002000000    A003000000  
0  5.673414e+11  2.046961e+10  
1  5.978178e+11  2.210983e+10  
2  5.939771e+11  3.042111e+10  

文件: income_stmt.csv
形状: (3, 5)
列名: ['Stkcd', 'ShortName', 'Accper', 'Typrep', 'B002000000']
   Stkcd ShortName      Accper Typrep    B002000000
0      1      深发展A  2010-01-01      A  5.030729e+09
1      1      深发展A  2010-03-31      A  1.578118e+09
2      1      深发展A  2010-06-30      A  3.033119e+09

文件: cashflow.csv
形状: (3, 9)
列名: ['Stkcd', 'ShortName', 'Accper', 'Typrep', 'D000103000', 'D000119000', 'D000120000', 'D000104000', 'D000105000']
   Stkcd Sh

In [60]:
# %% [cell 1] 导入库与路径设置
import pandas as pd
import numpy as np
import os

# 路径
raw = r"C:\Users\mingar\Downloads\Finance\homework\ex03\data_raw"
out = r"C:\Users\mingar\Downloads\Finance\homework\ex03\data_clean"
os.makedirs(out, exist_ok=True)

# 显示所有列（方便检查）
pd.set_option('display.max_columns', 50)

In [61]:
# %% [cell 2] ========== 1. 资产负债表 ==========
bs = pd.read_csv(os.path.join(raw, "balance_sheet.csv"))
# 只保留合并报表
bs = bs[bs['Typrep'] == 'A'].copy()
# 日期处理
bs['Accper'] = pd.to_datetime(bs['Accper'])
bs['year'] = bs['Accper'].dt.year
# 每个公司-年只保留最后一天（即年报12-31）
bs = bs.sort_values(['Stkcd', 'year', 'Accper'])
bs = bs.drop_duplicates(subset=['Stkcd', 'year'], keep='last')
# 保留并重命名需要的列
bs = bs[['Stkcd', 'year', 'ShortName',
         'A001000000', 'A002000000', 'A003000000', 'A001212000']].copy()
bs.columns = ['stkcd', 'year', 'name',
              'total_assets', 'total_liabilities', 'total_equity', 'fixed_assets']
print("资产负债表年频观测数:", bs.shape[0], "公司数:", bs['stkcd'].nunique())

资产负债表年频观测数: 61188 公司数: 5731


In [62]:
# %% [cell 3] ========== 2. 利润表 ==========
inc = pd.read_csv(os.path.join(raw, "income_stmt.csv"))
inc = inc[inc['Typrep'] == 'A'].copy()
inc['Accper'] = pd.to_datetime(inc['Accper'])
inc['year'] = inc['Accper'].dt.year
inc = inc.sort_values(['Stkcd', 'year', 'Accper'])
inc = inc.drop_duplicates(subset=['Stkcd', 'year'], keep='last')
inc = inc[['Stkcd', 'year', 'B002000000']].copy()
inc.columns = ['stkcd', 'year', 'net_profit']
print("利润表年频观测数:", inc.shape[0], "公司数:", inc['stkcd'].nunique())

利润表年频观测数: 61188 公司数: 5731


In [63]:
# %% [cell 4] ========== 3. 现金流量表 ==========
cf = pd.read_csv(os.path.join(raw, "cashflow.csv"))
cf = cf[cf['Typrep'] == 'A'].copy()
cf['Accper'] = pd.to_datetime(cf['Accper'])
cf['year'] = cf['Accper'].dt.year
cf = cf.sort_values(['Stkcd', 'year', 'Accper'])
cf = cf.drop_duplicates(subset=['Stkcd', 'year'], keep='last')
# 折旧摊销五项，缺失填0
dep_cols = ['D000103000', 'D000119000', 'D000120000', 'D000104000', 'D000105000']
for c in dep_cols:
    cf[c] = cf[c].fillna(0)
cf['dep_amort'] = cf[dep_cols].sum(axis=1)
cf = cf[['Stkcd', 'year', 'dep_amort']].copy()
cf.columns = ['stkcd', 'year', 'dep_amort']
print("现金流量表年频观测数:", cf.shape[0], "公司数:", cf['stkcd'].nunique())

现金流量表年频观测数: 61027 公司数: 5725


In [64]:
# %% [cell 5] ========== 4. ST/PT 标记整合 ==========
st_files = [
    "st_flag_2010-2014.csv",
    "st_flag_2015-2019.csv",
    "st_flag_2020-2024.csv",
    "st_flag_2025.csv"
]
st_list = []
for f in st_files:
    df = pd.read_csv(os.path.join(raw, f))
    st_list.append(df)
st_all = pd.concat(st_list, ignore_index=True)
# 提取年份
st_all['Trddt'] = pd.to_datetime(st_all['Trddt'])
st_all['year'] = st_all['Trddt'].dt.year
# 任何非正常交易状态（Trdsta != 1）视为特殊处理
st_abnormal = st_all[st_all['Trdsta'] != 1]
# 被特殊处理过的公司列表
st_firms = st_abnormal['Stkcd'].unique().tolist()
print("曾处于ST/*ST/PT的公司数量:", len(st_firms))

曾处于ST/*ST/PT的公司数量: 466


In [65]:
# %% [cell 6] ========== 5. 行业分类 ==========
ind = pd.read_csv(os.path.join(raw, "industry.csv"))
ind = ind[['Symbol', 'EndDate', 'IndustryCodeC']].copy()
ind.columns = ['stkcd', 'date', 'ind_code']
ind['date'] = pd.to_datetime(ind['date'])
ind['year'] = ind['date'].dt.year
# 向前填充缺失的行业代码（2023年后为空）
ind = ind.sort_values(['stkcd', 'year'])
ind['ind_code'] = ind.groupby('stkcd')['ind_code'].ffill()
# 仅保留每年的最后一条行业记录
ind = ind.drop_duplicates(subset=['stkcd', 'year'], keep='last')
ind = ind[['stkcd', 'year', 'ind_code']]
print("行业分类年频观测数:", ind.shape[0])

行业分类年频观测数: 56711


In [66]:
en = pd.read_csv(r"C:\Users\mingar\Downloads\Finance\homework\ex03\data_raw\EN_EquityNatureAll.csv")
print("列名:", en.columns.tolist())
print("\nEquityNatureID 所有唯一值:")
print(sorted(en['EquityNatureID'].dropna().unique()))
print("\n每种编码的观测数:")
print(en['EquityNatureID'].value_counts().sort_index())

列名: ['Symbol', 'ShortName', 'EndDate', 'EquityNatureID']

EquityNatureID 所有唯一值:
['1', '1,2', '1,2,3', '1,3', '2', '2,3', '3', '4']

每种编码的观测数:
EquityNatureID
1        17938
1,2         26
1,2,3        3
1,3         13
2        32212
2,3       1224
3         1970
4          806
Name: count, dtype: int64


In [67]:
# %% [cell 7] ========== 6. 股权性质（SOE）——严格纯国企/纯民营 ==========
own = pd.read_csv(os.path.join(raw, "EN_EquityNatureAll.csv"))
own = own[['Symbol', 'EndDate', 'EquityNatureID']].copy()
own.columns = ['stkcd', 'date', 'eq_nature']

# 日期格式：如 '31/12/2010'
own['date'] = pd.to_datetime(own['date'], dayfirst=True)
own['year'] = own['date'].dt.year
own = own.drop_duplicates(subset=['stkcd', 'year'], keep='last')

# 只保留纯国企(1)和纯民营(2)
own = own[own['eq_nature'].isin(['1', '2'])].copy()

# 生成 SOE 虚拟变量
own['soe'] = (own['eq_nature'] == '1').astype(int)

own = own[['stkcd', 'year', 'soe']]
print("股权性质（纯国企/纯民营）年频观测数:", own.shape[0], "公司数:", own['stkcd'].nunique())

股权性质（纯国企/纯民营）年频观测数: 50150 公司数: 5249


C:\Users\mingar\AppData\Local\Temp\ipykernel_31992\3043713747.py:7: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  own['date'] = pd.to_datetime(own['date'], dayfirst=True)


In [68]:
# %% [cell 8] ========== 7. 宏观M2 ==========
m2 = pd.read_csv(os.path.join(raw, "m2.csv"))
m2.columns = ['year', 'm2']
# 计算M2增长率（%）
m2 = m2.sort_values('year')
m2['m2_growth'] = m2['m2'].pct_change() * 100
m2 = m2[['year', 'm2_growth']]
print("M2数据年份范围:", m2['year'].min(), "-", m2['year'].max())

M2数据年份范围: 2010 - 2024


In [69]:
# %% [cell 9] ========== 8. 合并所有表 ==========
# 以资产负债表为起点
df = bs.merge(inc, on=['stkcd', 'year'], how='left')
df = df.merge(cf, on=['stkcd', 'year'], how='left')
df = df.merge(ind, on=['stkcd', 'year'], how='left')
df = df.merge(own, on=['stkcd', 'year'], how='left')
# 合并宏观数据
df = df.merge(m2, on='year', how='left')

print("合并后观测数:", df.shape[0], "公司数:", df['stkcd'].nunique())

合并后观测数: 61188 公司数: 5731


In [70]:
# %% [cell 10] ========== 9. 变量构造 ==========
# NPR = 净利润 / 总资产
df['npr'] = df['net_profit'] / df['total_assets']
# Lev
df['lev'] = df['total_liabilities'] / df['total_assets']
# Size
df['size'] = np.log(df['total_assets'])
# Tangibility
df['tang'] = df['fixed_assets'] / df['total_assets']
# NDTS
df['ndts'] = df['dep_amort'] / df['total_assets']
# Growth: 需要滞后 total_assets
df = df.sort_values(['stkcd', 'year'])
df['total_assets_lag'] = df.groupby('stkcd')['total_assets'].shift(1)
df['growth'] = (df['total_assets'] - df['total_assets_lag']) / df['total_assets_lag']
# 行业代码生成：制造业取前两位(C13)，其他取首字母(J)
df['ind_code'] = df['ind_code'].fillna('Z')  # 防止NaN
df['ind_sector'] = df['ind_code'].apply(lambda x: x[:3] if x.startswith('C') else x[0])
# 剔除行业代码为Z的（即完全缺失）
df = df[df['ind_sector'] != 'Z']
print("变量构造后观测数:", df.shape[0])

变量构造后观测数: 55915


c:\Users\mingar\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [71]:
# %% [cell 11] ========== 10. 样本筛选 ==========
def record_step(data, step_desc, step_type='drop'):
    print(f"--- {step_desc} ---")
    print(f"当前观测数: {data.shape[0]}, 公司数: {data['stkcd'].nunique()}")
    return data

df = record_step(df, "初始样本（合并后）")

# (1) 剔除金融、保险行业 (J开头)
df = df[~df['ind_sector'].str.startswith('J')]
df = record_step(df, "剔除金融保险行业")

# (2) 剔除曾被ST/PT的公司
df = df[~df['stkcd'].isin(st_firms)]
df = record_step(df, "剔除曾被ST/PT的公司")

# (3) 剔除资不抵债 (Lev > 1)
df = df[df['lev'] <= 1]
df = record_step(df, "剔除资产负债率 > 1")

# (4) 剔除关键变量缺失
key_vars = ['lev', 'npr', 'size', 'tang', 'growth', 'ndts', 'soe']
df = df.dropna(subset=key_vars)
df = record_step(df, "剔除关键变量缺失")

# (5) 年份范围限制在2010-2025
df = df[(df['year'] >= 2010) & (df['year'] <= 2025)]
df = record_step(df, "保留2010-2025年")

df_final = df.copy()

--- 初始样本（合并后） ---
当前观测数: 55915, 公司数: 5269
--- 剔除金融保险行业 ---
当前观测数: 54609, 公司数: 5173
--- 剔除曾被ST/PT的公司 ---
当前观测数: 48080, 公司数: 4708
--- 剔除资产负债率 > 1 ---
当前观测数: 47883, 公司数: 4708
--- 剔除关键变量缺失 ---
当前观测数: 38385, 公司数: 4357
--- 保留2010-2025年 ---
当前观测数: 38385, 公司数: 4357


In [72]:
# %% [cell 12] ========== 11. Winsorize (逐年双侧1%) ==========
winsor_vars = ['lev', 'npr', 'tang', 'growth', 'ndts']
for var in winsor_vars:
    df_final[var] = df_final.groupby('year')[var].transform(
        lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99))
    )
print("Winsorize完成")

Winsorize完成


In [73]:
# %% [cell 13] ========== 12. 保存清洗后数据 ==========
# 选择需要的列保存
final_cols = ['stkcd', 'year', 'name', 'lev', 'npr', 'size', 'tang', 'growth',
              'ndts', 'soe', 'ind_sector', 'm2_growth']
df_final[final_cols].to_csv(os.path.join(out, "analysis_data.csv"), index=False)
print(f"最终样本: {df_final.shape[0]} 个观测, {df_final['stkcd'].nunique()} 家公司")
print("清洗后数据已保存至 data_clean/analysis_data.csv")

最终样本: 38385 个观测, 4357 家公司
清洗后数据已保存至 data_clean/analysis_data.csv


In [74]:
import pandas as pd
import os

clean_dir = r"C:\Users\mingar\Downloads\Finance\homework\ex03\data_clean"
df = pd.read_csv(os.path.join(clean_dir, "analysis_data.csv"))

# 1. 前5行样例
print("========== 数据样例 ==========")
print(df.head())

# 2. 列名和缺失值
print("\n========== 列信息 ==========")
print(df.info())

# 3. 年份和公司数
print("\n========== 样本概况 ==========")
print(f"观测数: {df.shape[0]}")
print(f"公司数: {df['stkcd'].nunique()}")
print(f"年份范围: {df['year'].min()} - {df['year'].max()}")

# 4. 主要变量统计（快速检查有无异常）
print("\n========== 主要变量描述 ==========")
print(df[['lev','npr','size','tang','growth','ndts','soe']].describe())

========== 数据样例 ==========
   stkcd  year name       lev       npr       size      tang    growth  \
0      2  2011  万科A  0.770997  0.039160  26.414329  0.005388  0.373640   
1      2  2012  万科A  0.783163  0.041348  26.660278  0.004256  0.278835   
2      2  2013  万科A  0.779970  0.038183  26.895395  0.004444  0.265056   
3      2  2014  万科A  0.772046  0.037937  26.954552  0.004540  0.060941   
4      2  2015  万科A  0.777015  0.042450  27.138846  0.008044  0.202370   

       ndts  soe ind_sector  m2_growth  
0  0.000711  1.0          K   8.711284  
1  0.000761  1.0          K   6.491868  
2  0.000881  1.0          K   9.274421  
3  0.001080  1.0          K   3.191712  
4  0.001171  1.0          K  15.197832  

========== 列信息 ==========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38385 entries, 0 to 38384
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   stkcd       38385 non-null  int64  
 1   year        38385 

In [76]:
df = pd.read_csv(r"C:\Users\mingar\Downloads\Finance\homework\ex03\data_clean\analysis_data.csv")
print(f"观测数: {df.shape[0]}, 公司数: {df['stkcd'].nunique()}")
print("SOE 均值:", df['soe'].mean())

观测数: 38385, 公司数: 4357
SOE 均值: 0.36449133776214665


In [77]:
df = pd.read_csv(r"C:\Users\mingar\Downloads\Finance\homework\ex03\data_clean\analysis_data.csv")
print(f"观测数: {df.shape[0]}, 公司数: {df['stkcd'].nunique()}")
print("SOE 均值:", df['soe'].mean())

观测数: 38385, 公司数: 4357
SOE 均值: 0.36449133776214665
